In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolTransforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DIM_F1 = 20
DIM_F2_RAW = 1024
DIM_F2_PCA = 128
DIM_F3 = 10
DIM_F4 = 43
DIM_HB = 2
DIM_ONE_ASSAY = 3
DIM_INTER = 14

MODAL_DIMS_SINGLE = [DIM_F1, DIM_F2_PCA, DIM_F3, DIM_F4, DIM_HB, DIM_ONE_ASSAY]
FUSION_HIDDEN = 192    
HA_HB_DIM = 192        
GAN_INPUT_DIM = HA_HB_DIM * 2 + DIM_INTER

EMBED_DIM = 32
EXP_DIM = 32
INPUT_DIM = GAN_INPUT_DIM + EMBED_DIM + EXP_DIM

ASSAY_COLS = [
    "SMILES1-assay1", "SMILES1-assay2", "SMILES1-assay3",
    "SMILES2-assay1", "SMILES2-assay2", "SMILES2-assay3"
]

BATCH_SIZE = 32
EPOCHS = 150
LR = 2e-4
PATIENCE = 30
NUM_CLASSES = 5
WEIGHT_DECAY = 1e-5

GAN_NOISE_DIM = 128
GAN_EPOCHS = 200      
GAN_BATCH_DEFAULT = 16
GAN_LR = 1e-4
GP_LAMBDA = 15        
CRITIC_ITER = 8       

cls_count = {0:929, 1:870, 2:897, 3:260, 4:195}
target_num = 929
need_gen = {c: target_num - cnt for c, cnt in cls_count.items()}
print("每类需要生成样本数：", need_gen)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = torch.tensor(alpha, dtype=torch.float32).to(device) if alpha else None

    def forward(self, inputs, targets):
        logpt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(logpt)
        logpt = (1 - pt) ** self.gamma * logpt
        if self.alpha is not None:
            logpt = self.alpha * logpt
        loss = F.nll_loss(logpt, targets, reduction=self.reduction)
        return loss

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim, lambda_c=0.001):
        super().__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.lambda_c = lambda_c
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim).to(device))

    def forward(self, feat, label):
        batch_size = feat.size(0)
        feat = feat.view(batch_size, -1)
        centers_batch = self.centers.index_select(0, label)
        dist = torch.sum((feat - centers_batch) ** 2, dim=1)
        return self.lambda_c * torch.mean(dist)

def to_fixed_len(arr, target_len):
    arr = np.asarray(arr, dtype=np.float32).ravel()
    if len(arr) < target_len:
        arr = np.concatenate([arr, np.zeros(target_len - len(arr), dtype=np.float32)])
    else:
        arr = arr[:target_len]
    return arr

def generate_3d_conformer(mol, max_attempts=10):
    try:
        mol_3d = Chem.AddHs(mol)
        for _ in range(max_attempts):
            if AllChem.EmbedMolecule(mol_3d, randomSeed=42) == 0:
                AllChem.MMFFOptimizeMolecule(mol_3d)
                return mol_3d
        return None
    except:
        return None

def get_3d_geometric_features(mol_3d):
    feats = []
    if mol_3d is not None and mol_3d.GetNumConformers() > 0:
        try:
            conf = mol_3d.GetConformer()
            num_atoms = mol_3d.GetNumAtoms()
            coords = np.array([[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
                               for i in range(num_atoms)])
            feats.extend(coords.mean(axis=0).tolist())
            feats.extend(coords.std(axis=0).tolist())
            feats.extend(coords.max(axis=0).tolist())
            feats.extend(coords.min(axis=0).tolist())

            bond_lengths = [rdMolTransforms.GetBondLength(conf, b.GetBeginAtomIdx(), b.GetEndAtomIdx())
                            for b in mol_3d.GetBonds()]
            feats += [np.mean(bond_lengths), np.std(bond_lengths), np.max(bond_lengths), np.min(bond_lengths)] if bond_lengths else [0.0]*4

            angles = []
            for a in range(num_atoms):
                for n in mol_3d.GetAtomWithIdx(a).GetNeighbors():
                    angles.append(rdMolTransforms.GetAngleDeg(conf, a, n.GetIdx(), a))
            feats += [np.mean(angles), np.std(angles), np.max(angles), np.min(angles)] if angles else [0.0]*4

            dihedrals = []
            for b in mol_3d.GetBonds():
                a1, a2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
                for n1 in mol_3d.GetAtomWithIdx(a1).GetNeighbors():
                    for n2 in mol_3d.GetAtomWithIdx(a2).GetNeighbors():
                        if n1.GetIdx() != a2 and n2.GetIdx() != a1:
                            dihedrals.append(rdMolTransforms.GetDihedralDeg(conf, n1.GetIdx(), a1, a2, n2.GetIdx()))
            feats += dihedrals[:20]
        except:
            pass
    return to_fixed_len(feats, DIM_F4)

def extract_mol_features_raw(smi):
    f1 = np.zeros(DIM_F1, dtype=np.float32)
    f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f3 = np.zeros(DIM_F3, dtype=np.float32)
    f4 = np.zeros(DIM_F4, dtype=np.float32)
    hb = np.zeros(DIM_HB, dtype=np.float32)

    if pd.isna(smi) or not isinstance(smi, str) or len(smi.strip()) == 0:
        return f1, f2_raw, f3, f4, hb

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return f1, f2_raw, f3, f4, hb

    f1 = np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumValenceElectrons(mol), Descriptors.HeavyAtomCount(mol),
        Descriptors.NHOHCount(mol), Descriptors.NOCount(mol), Descriptors.FractionCSP3(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
        Descriptors.NumSaturatedRings(mol), Descriptors.NumHeteroatoms(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.MaxPartialCharge(mol) if Descriptors.MaxPartialCharge(mol) else 0.0,
        Descriptors.MinPartialCharge(mol) if Descriptors.MinPartialCharge(mol) else 0.0,
        Descriptors.MolMR(mol), 0.0, 0.0,
    ], dtype=np.float32)
    f1 = to_fixed_len(f1, DIM_F1)

    try:
        fp_bit = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=DIM_F2_RAW)
        f2_raw = np.array(fp_bit, dtype=np.float32)
    except:
        f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f2_raw = to_fixed_len(f2_raw, DIM_F2_RAW)

    f3 = np.array([
        Descriptors.MolWt(mol) * 0.1, Descriptors.MolLogP(mol) * 0.5, Descriptors.TPSA(mol) * 0.01,
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
        Descriptors.NumHeteroatoms(mol), Descriptors.NumRotatableBonds(mol),
        Descriptors.LabuteASA(mol) if Descriptors.LabuteASA(mol) else 0.0,
    ], dtype=np.float32)
    f3 = to_fixed_len(f3, DIM_F3)

    mol_3d = generate_3d_conformer(mol)
    f4 = get_3d_geometric_features(mol_3d)

    h_don = Descriptors.NumHDonors(mol)
    h_acc = Descriptors.NumHAcceptors(mol)
    hb = np.array([h_don, h_acc], dtype=np.float32)
    hb = to_fixed_len(hb, DIM_HB)

    return f1, f2_raw, f3, f4, hb

def calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b):
    def cos_sim(u, v):
        n1, n2 = np.linalg.norm(u), np.linalg.norm(v)
        return 0.0 if (n1 < 1e-8 or n2 < 1e-8) else np.dot(u, v) / (n1 * n2)
    def l2_dist(u, v):
        return np.linalg.norm(u - v)

    cos1 = cos_sim(f1a, f1b)
    dist1 = l2_dist(f1a, f1b)
    had1 = (f1a * f1b).mean()

    cos3 = cos_sim(f4a, f4b)
    dist3 = l2_dist(f4a, f4b)
    had3 = (f4a * f4b).mean()

    had_low = (f3a * f3b).mean()

    da, aa = hb_a[0], hb_a[1]
    db, ab = hb_b[0], hb_b[1]
    sum_d, sum_a = da + db, aa + ab
    diff_d, diff_a = abs(da - db), abs(aa - ab)
    mul_d, mul_a = da * db, aa * ab

    inter = np.array([
        cos1, dist1, had1, cos3, dist3, had3, had_low,
        sum_d, sum_a, diff_d, diff_a, mul_d, mul_a, 0.0
    ], dtype=np.float32)
    return to_fixed_len(inter, DIM_INTER)

class CrossInteractAttnFusion(nn.Module):
    def __init__(self, modal_dims, hidden_dim, out_dim, head=4):
        super().__init__()
        self.modal_dims = modal_dims
        self.num_modal = len(modal_dims)
        self.head = head
        self.head_dim = hidden_dim // head
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim

        self.projs = nn.ModuleList([nn.Linear(d, hidden_dim) for d in modal_dims])
        self.ln_proj = nn.LayerNorm(hidden_dim)

        self.qkv_global = nn.Linear(hidden_dim, hidden_dim * 3)

        self.assay_gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Sigmoid()
        )

        self.cross_q = nn.Linear(hidden_dim, hidden_dim)
        self.cross_k = nn.Linear(hidden_dim, hidden_dim)
        self.cross_v = nn.Linear(hidden_dim, hidden_dim)
        self.ln_cross = nn.LayerNorm(hidden_dim)

        self.concat_ln = nn.LayerNorm(hidden_dim * self.num_modal)
        self.out_linear = nn.Linear(hidden_dim * self.num_modal, out_dim)
        self.drop = nn.Dropout(0.35)

    def forward(self, modal_list):
        bs = modal_list[0].shape[0]
        hidden_list = []
        for idx, m in enumerate(modal_list):
            h = self.projs[idx](m)
            h = self.ln_proj(h)
            hidden_list.append(h)
        stack = torch.stack(hidden_list, dim=1)

        qkv = self.qkv_global(stack).reshape(bs, self.num_modal, 3, self.head, self.head_dim).permute(2,0,3,1,4)
        q_g, k_g, v_g = qkv[0], qkv[1], qkv[2]
        attn_score = torch.matmul(q_g, k_g.transpose(-1,-2)) / np.sqrt(self.head_dim)
        attn_weight = F.softmax(attn_score, dim=-1)
        global_attn_out = torch.matmul(attn_weight, v_g).transpose(1,2).reshape(bs, self.num_modal, self.hidden_dim)

        assay_feat = stack[:, -1, :]
        gate_w = self.assay_gate(assay_feat).unsqueeze(1)
        gated_feat = global_attn_out * gate_w

        molA_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        molB_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        q_c = self.cross_q(molA_feat)
        k_c = self.cross_k(molB_feat)
        v_c = self.cross_v(molB_feat)
        cross_attn = F.softmax(torch.matmul(q_c, k_c.transpose(-1,-2)) / np.sqrt(self.head_dim), dim=-1)
        cross_out = torch.matmul(cross_attn, v_c)
        cross_out = self.ln_cross(cross_out)
        fused_all = gated_feat + cross_out

        concat = fused_all.reshape(bs, -1)
        concat = self.concat_ln(concat)
        out = self.drop(self.out_linear(concat))
        return out

class ToxicityDataset(Dataset):
    def __init__(self, feat, species, exposure, label):
        self.feat = feat
        self.species = species
        self.exposure = exposure
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.feat[idx]).float(),
            torch.tensor(self.species[idx], dtype=torch.long),
            torch.tensor([self.exposure[idx]], dtype=torch.float32),
            torch.tensor(self.label[idx], dtype=torch.long)
        )

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.4):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        residual = x
        out = self.dropout(self.act(self.bn1(self.fc1(x))))
        out = self.bn2(self.fc2(out))
        out += residual
        return self.act(out)

class ImprovedToxicModel(nn.Module):
    def __init__(self, total_feat_dim, embed_dim, exp_dim, species_num, num_classes):
        super().__init__()
        self.embed_dim = embed_dim
        self.exp_dim = exp_dim
        self.total_feat_dim = total_feat_dim
        self.species_emb = nn.Embedding(species_num, embed_dim)
        self.exp_proj = nn.Linear(1, exp_dim)
        self.backbone = nn.Sequential(
            nn.Linear(INPUT_DIM, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(1024),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(512),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            ResBlock(256),
        )
        self.classifier = nn.Linear(256, num_classes)
        self.feature_out_dim = 256

    def forward(self, full_feat, species, exposure):
        sp_feat = self.species_emb(species)
        exp_feat = F.gelu(self.exp_proj(exposure)).squeeze(1)
        all_feat = torch.cat([full_feat, sp_feat, exp_feat], dim=-1)
        hidden = self.backbone(all_feat)
        logits = self.classifier(hidden)
        return logits, hidden

class WGANGP_Generator(nn.Module):
    def __init__(self, noise_dim, cond_dim, feat_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 1024),
            nn.LayerNorm(1024),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.25),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.25),
            nn.Linear(512, feat_dim),
        )
    def forward(self, z, cond):
        return self.net(torch.cat([z, cond], dim=1))

class WGANGP_Critic(nn.Module):
    def __init__(self, feat_dim, cond_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim + cond_dim, 512),
            nn.LeakyReLU(0.2, True),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.LeakyReLU(0.2, True),
            nn.Linear(256, 1),
        )
    def forward(self, feat, cond):
        return self.net(torch.cat([feat, cond], dim=1))

def compute_gp(critic, real, fake, cond):
    bs = real.shape[0]
    alpha = torch.rand(bs, 1).to(device)
    interp = (alpha * real + (1 - alpha) * fake).clone().detach()
    interp.requires_grad = True
    out = critic(interp, cond)
    grad = torch.autograd.grad(outputs=out.sum(), inputs=interp, create_graph=True)[0]
    grad_norm = grad.norm(2, dim=1)
    return torch.mean((grad_norm - 1) ** 2)

def train_wgangp(X, y, num_classes, noise_dim, epochs, batch_size_max, lr):
    feat_dim = X.shape[1]
    sample_num = X.shape[0]
    batch_size = min(batch_size_max, sample_num)
    y_onehot = np.eye(num_classes)[y]
    X_tensor = torch.from_numpy(X).float().to(device)
    y_tensor = torch.from_numpy(y_onehot).float().to(device)
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    G = WGANGP_Generator(noise_dim, num_classes, feat_dim).to(device)
    C = WGANGP_Critic(feat_dim, num_classes).to(device)
    opt_G = Adam(G.parameters(), lr=lr, betas=(0.0,0.9))
    opt_C = Adam(C.parameters(), lr=lr, betas=(0.0,0.9))
    print("开始训练当前类 WGAN-GP（强化模式防崩溃配置）...")
    for epoch in range(epochs):
        G.train()
        C.train()
        total_c_loss = 0.0
        total_g_loss = 0.0
        for real_feat, real_cond in dataloader:
            bsz = real_feat.size(0)
            for _ in range(CRITIC_ITER):
                opt_C.zero_grad()
                real_out = C(real_feat, real_cond)
                z = torch.randn(bsz, noise_dim, device=device)
                fake_feat = G(z, real_cond)
                fake_out = C(fake_feat.detach(), real_cond)
                gp = compute_gp(C, real_feat, fake_feat, real_cond)
                c_loss = torch.mean(fake_out) - torch.mean(real_out) + GP_LAMBDA * gp
                c_loss.backward()
                opt_C.step()
            total_c_loss += c_loss.item()
            opt_G.zero_grad()
            z = torch.randn(bsz, noise_dim, device=device)
            fake_feat = G(z, real_cond)
            fake_out = C(fake_feat, real_cond)
            g_loss = -torch.mean(fake_out)
            g_loss.backward()
            opt_G.step()
            total_g_loss += g_loss.item()
        total_c_loss /= len(dataloader) * CRITIC_ITER
        total_g_loss /= len(dataloader)
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:2d} | Critic Loss: {total_c_loss:.4f} | Gen Loss: {total_g_loss:.4f}")
    return G

def generate_by_class(G, cls, gen_num, num_classes, noise_dim):
    G.eval()
    cond = torch.zeros(gen_num, num_classes, device=device)
    cond[:, cls] = 1.0
    z = torch.randn(gen_num, noise_dim, device=device)
    with torch.no_grad():
        fake_feat = G(z, cond)
        noise = torch.normal(mean=0.0, std=0.01, size=fake_feat.shape, device=device)
        fake_feat = fake_feat + noise
    return fake_feat.cpu().numpy(), np.full(gen_num, cls)

if __name__ == "__main__":
    FILE_PATH = "reproduction-toxicity-traindata-Enumerator.xlsx"  #neurotoxicity/immunotoxicity/hepatotoxicity/developmental-toxicity-traindata-Enumerator.xlsx
    df = pd.read_excel(FILE_PATH, sheet_name=0)
    df["Interaction types of compounds"] = df["Interaction types of compounds"].str.strip()
    label_map = {
        "Antagonistic effect": 0,
        "Synergistic effect": 1,
        "Not mentioned": 2,
        "Additive effect": 3,
        "Independent effect": 4
    }
    df = df[df["Interaction types of compounds"].isin(label_map.keys())].reset_index(drop=True)
    df["label"] = df["Interaction types of compounds"].map(label_map)
    print("原始标签分布：")
    print(df["Interaction types of compounds"].value_counts())

    df["species"] = df["species"].fillna("Unknown")
    sp_enc = LabelEncoder()
    df["sp_enc"] = sp_enc.fit_transform(df["species"])
    num_sp = len(sp_enc.classes_)

    print("\n收集指纹并训练PCA...")
    all_fp_raw = []
    for _, row in df.iterrows():
        _, f2a_raw, _, _, _ = extract_mol_features_raw(row["SMILES1"])
        _, f2b_raw, _, _, _ = extract_mol_features_raw(row["SMILES2"])
        all_fp_raw.append(f2a_raw)
        all_fp_raw.append(f2b_raw)
    all_fp_raw = np.array(all_fp_raw, dtype=np.float32)
    pca = PCA(n_components=DIM_F2_PCA, random_state=42)
    pca.fit(all_fp_raw)
    print(f"PCA累计方差解释率: {np.sum(pca.explained_variance_ratio_):.4f}")

    fusion_model = CrossInteractAttnFusion(MODAL_DIMS_SINGLE, FUSION_HIDDEN, HA_HB_DIM).to(device)
    fusion_model.eval()

    print(f"\n单分子融合输出HA/HB维度: {HA_HB_DIM}")
    print(f"GAN输入基础特征维度(HA+HB+inter): {GAN_INPUT_DIM}")
    X_list = []
    sp_list = []
    exp_list = []
    lab_list = []

    for _, row in df.iterrows():
        f1a, f2a_raw, f3a, f4a, hb_a = extract_mol_features_raw(row["SMILES1"])
        f1b, f2b_raw, f3b, f4b, hb_b = extract_mol_features_raw(row["SMILES2"])
        f2a = pca.transform(f2a_raw.reshape(1, -1))[0]
        f2b = pca.transform(f2b_raw.reshape(1, -1))[0]
        assayA = row[["SMILES1-assay1","SMILES1-assay2","SMILES1-assay3"]].values.astype(np.float32)
        assayB = row[["SMILES2-assay1","SMILES2-assay2","SMILES2-assay3"]].values.astype(np.float32)
        inter_feat = calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b)
        modA = [
            torch.from_numpy(f1a).float().unsqueeze(0).to(device),
            torch.from_numpy(f2a).float().unsqueeze(0).to(device),
            torch.from_numpy(f3a).float().unsqueeze(0).to(device),
            torch.from_numpy(f4a).float().unsqueeze(0).to(device),
            torch.from_numpy(hb_a).float().unsqueeze(0).to(device),
            torch.from_numpy(assayA).float().unsqueeze(0).to(device)
        ]
        modB = [
            torch.from_numpy(f1b).float().unsqueeze(0).to(device),
            torch.from_numpy(f2b).float().unsqueeze(0).to(device),
            torch.from_numpy(f3b).float().unsqueeze(0).to(device),
            torch.from_numpy(f4b).float().unsqueeze(0).to(device),
            torch.from_numpy(hb_b).float().unsqueeze(0).to(device),
            torch.from_numpy(assayB).float().unsqueeze(0).to(device)
        ]
        with torch.no_grad():
            HA = fusion_model(modA).squeeze(0).cpu().numpy()
            HB = fusion_model(modB).squeeze(0).cpu().numpy()
        gan_base_feat = np.concatenate([HA, HB, inter_feat])
        X_list.append(gan_base_feat)
        sp_list.append(row["sp_enc"])
        exp_list.append(row["exposure duration (days)"])
        lab_list.append(row["label"])

    X_raw = np.stack(X_list, axis=0)
    sp_arr = np.array(sp_list)
    exp_arr = np.array(exp_list, dtype=np.float32).reshape(-1, 1)
    y_arr = np.array(lab_list)
    print(f"融合后GAN基础特征矩阵shape: {X_raw.shape}")

    feat_imp = SimpleImputer(strategy="mean")
    X_raw = feat_imp.fit_transform(X_raw)
    exp_imp = SimpleImputer(strategy="mean")
    exp_arr = exp_imp.fit_transform(exp_arr)
    scaler = StandardScaler()
    X_raw = scaler.fit_transform(X_raw)
    exp_scaler = StandardScaler()
    exp_arr = exp_scaler.fit_transform(exp_arr)

    X_train, X_test, sp_train, sp_test, exp_train, exp_test, y_train, y_test = \
        train_test_split(X_raw, sp_arr, exp_arr, y_arr, test_size=0.2,
                         random_state=42, stratify=y_arr)

    print("\n===== 开始按类别WGAN-GP生成融合基础特征（强化防模式崩溃配置） =====")
    gen_X_total = []
    gen_y_total = []
    gen_sp_total = []
    gen_exp_total = []

    for cls in range(NUM_CLASSES):
        need = need_gen[cls]
        if need <= 0:
            print(f"类别 {cls} 样本充足，无需生成")
            continue
        idx = y_train == cls
        X_cls = X_train[idx]
        y_cls = y_train[idx]
        print(f"\n类别 {cls}：现有{len(X_cls)}条，需生成 {need} 条样本")
        G = train_wgangp(X_cls, y_cls, NUM_CLASSES, GAN_NOISE_DIM, GAN_EPOCHS, GAN_BATCH_DEFAULT, GAN_LR)
        g_feat, g_lab = generate_by_class(G, cls, need, NUM_CLASSES, GAN_NOISE_DIM)
        gen_X_total.append(g_feat)
        gen_y_total.append(g_lab)
        sp_cls = sp_train[idx]
        exp_cls = exp_train[idx]
        g_sp = np.random.choice(sp_cls, size=need)
        g_exp = exp_cls[np.random.choice(len(exp_cls), size=need)]
        gen_sp_total.append(g_sp)
        gen_exp_total.append(g_exp)

    if gen_X_total:
        gen_X = np.vstack(gen_X_total)
        gen_y = np.hstack(gen_y_total)
        gen_sp = np.hstack(gen_sp_total)
        gen_exp = np.vstack(gen_exp_total)
        X_train_bal = np.concatenate([X_train, gen_X], axis=0)
        y_train_bal = np.concatenate([y_train, gen_y], axis=0)
        sp_train_bal = np.concatenate([sp_train, gen_sp], axis=0)
        exp_train_bal = np.concatenate([exp_train, gen_exp], axis=0)
    else:
        X_train_bal = X_train
        y_train_bal = y_train
        sp_train_bal = sp_train
        exp_train_bal = exp_train

    print("\n均衡后训练集各类数量：")
    unique, counts = np.unique(y_train_bal, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"类别 {u}: {c}")

    sample_weights = np.zeros_like(y_train_bal, dtype=np.float32)
    for cls, cnt in zip(unique, counts):
        sample_weights[y_train_bal == cls] = 1.0 / cnt
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    train_set = ToxicityDataset(X_train_bal, sp_train_bal, exp_train_bal, y_train_bal)
    test_set = ToxicityDataset(X_test, sp_test, exp_test, y_test)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = ImprovedToxicModel(
        total_feat_dim=GAN_INPUT_DIM,
        embed_dim=EMBED_DIM,
        exp_dim=EXP_DIM,
        species_num=num_sp,
        num_classes=NUM_CLASSES
    ).to(device)

    total = sum(cls_count.values())
    class_alpha = [total / cls_count[i] for i in range(NUM_CLASSES)]
    focal_loss = FocalLoss(alpha=class_alpha, gamma=2.0)
    center_loss = CenterLoss(num_classes=NUM_CLASSES, feat_dim=model.feature_out_dim)
    optimizer = Adam(list(model.parameters()) + list(center_loss.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    best_acc = 0.0
    best_epoch = 0
    stop_cnt = 0
    print("\n===== 开始训练均衡数据集优化分类模型（WGAN-GP生成质量强化） =====")

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            ft, sp, exp, lab = [x.to(device) for x in batch]
            logits, hidden_feat = model(ft, sp, exp)
            loss_focal = focal_loss(logits, lab)
            loss_center = center_loss(hidden_feat, lab)
            loss = loss_focal + loss_center
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                ft, sp, exp, lab = [x.to(device) for x in batch]
                logits, hidden_feat = model(ft, sp, exp)
                loss_focal = focal_loss(logits, lab)
                loss_center = center_loss(hidden_feat, lab)
                val_loss += (loss_focal + loss_center).item()
                pred = torch.argmax(logits, dim=1)
                preds.extend(pred.cpu().numpy())
                trues.extend(lab.cpu())
        val_loss /= len(test_loader)
        acc = accuracy_score(trues, preds)

        if acc > best_acc:
            best_acc = acc
            best_epoch = epoch
            stop_cnt = 0
            torch.save({
                "model": model.state_dict(),
                "scaler": scaler,
                "exp_scaler": exp_scaler,
                "sp_enc": sp_enc,
                "label_map": label_map,
                "pca": pca,
                "fusion": fusion_model.state_dict()
            }, "reproduction_model.pth")   #neurotoxicity/immunotoxicity/hepatotoxicity/developmental-toxicity_model.pth
        else:
            stop_cnt += 1
            if stop_cnt >= PATIENCE:
                print(f"早停触发，最优轮次: {best_epoch+1}")
                break
        print(f"E{epoch+1:2d} | TrainLoss:{train_loss:.4f} | ValLoss:{val_loss:.4f} | Acc:{acc:.4f} | Best:{best_acc:.4f}")

    print("\n===== WGAN-GP生成强化版模型最终评估 =====")
    ckpt = torch.load("reproduction_model.pth", map_location=device) #neurotoxicity/immunotoxicity/hepatotoxicity/developmental-toxicity_model.pth
    model.load_state_dict(ckpt["model"])
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in test_loader:
            ft, sp, exp, lab = [x.to(device) for x in batch]
            logits, _ = model(ft, sp, exp)
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(lab.cpu())

    report = classification_report(trues, preds, target_names=list(label_map.keys()), digits=4)
    print(report)
    print(f"最优准确率: {best_acc:.4f}")

In [ ]:
#SHAP
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import os

save_dir = "reproduction_SHAP1"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Folder '{save_dir}' created automatically.")

DIM_F1 = 20
DIM_F2_PCA = 128
DIM_F3 = 10
DIM_F4 = 43
DIM_HB = 2
DIM_ONE_ASSAY = 3
MODAL_DIMS_SINGLE = [DIM_F1, DIM_F2_PCA, DIM_F3, DIM_F4, DIM_HB, DIM_ONE_ASSAY]
INNER_MODAL_NAMES = [
    "F1_BasicPhysicochemical(20D)",
    "F2_PCA_MorganFingerprint(128D)",
    "F3_ScaledPhysicochemical(10D)",
    "F4_3DGeometry(43D)",
    "HB_HydrogenBondStats(2D)",
    "Assay_InVitroToxicity(3D)"
]
EXTRA_MODAL_NAMES = [
    "Inter_MoleculeInteraction(14D)",
    "Species_ExperimentalCode",
    "Exposure_Duration"
]
ALL_MODAL_NAMES = INNER_MODAL_NAMES + EXTRA_MODAL_NAMES

HA_HB_DIM = 192
DIM_INTER = 14
GAN_INPUT_DIM = HA_HB_DIM * 2 + DIM_INTER
TOTAL_INPUT_DIM = GAN_INPUT_DIM + 2
NUM_CLASSES = 5

cls_desc = {
    0: "Antagonistic_effect",
    1: "Synergistic_effect",
    2: "Unspecified_interaction",
    3: "Additive_effect",
    4: "Independent_effect"
}

SHAP_SAMPLE_NUM = 100
TARGET_CLS = 4
TOP_K_FEAT = 20

def load_fusion_weight_breakdown(device, fusion_path="reproduction_model.pth"):  #neurotoxicity/immunotoxicity/hepatotoxicity/developmental-toxicity_model.pth
    ckpt = torch.load(fusion_path, map_location=device)
    from torch import nn
    class CrossInteractAttnFusion(nn.Module):
        def __init__(self, modal_dims, hidden_dim, out_dim, head=4):
            super().__init__()
            self.modal_dims = modal_dims
            self.num_modal = len(modal_dims)
            self.head = head
            self.head_dim = hidden_dim // head
            self.hidden_dim = hidden_dim
            self.out_dim = out_dim
            self.projs = nn.ModuleList([nn.Linear(d, hidden_dim) for d in modal_dims])
            self.ln_proj = nn.LayerNorm(hidden_dim)
            self.qkv_global = nn.Linear(hidden_dim, hidden_dim * 3)
            self.assay_gate = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.Sigmoid()
            )
            self.cross_q = nn.Linear(hidden_dim, hidden_dim)
            self.cross_k = nn.Linear(hidden_dim, hidden_dim)
            self.cross_v = nn.Linear(hidden_dim, hidden_dim)
            self.ln_cross = nn.LayerNorm(hidden_dim)
            self.concat_ln = nn.LayerNorm(hidden_dim * self.num_modal)
            self.out_linear = nn.Linear(hidden_dim * self.num_modal, out_dim)
            self.drop = nn.Dropout(0.35)
    fusion_model = CrossInteractAttnFusion(MODAL_DIMS_SINGLE, hidden_dim=192, out_dim=192).to(device)
    fusion_model.load_state_dict(ckpt["fusion"])
    fusion_model.eval()

    modal_proj_weights = []
    for proj in fusion_model.projs:
        w = proj.weight.detach().cpu().numpy()
        modal_proj_weights.append(np.abs(w).mean(axis=1))
    modal_to_hidden = np.stack(modal_proj_weights, axis=0)
    return modal_to_hidden

class GradSHAPExplainer:
    def __init__(self, model, X_test, sp_test, exp_test, device, sample_num=20):
        self.model = model
        self.model.eval()
        self.device = device
        self.sample_num = sample_num
        self.X = X_test[:sample_num]
        self.sp = sp_test[:sample_num]
        self.exp = exp_test[:sample_num]
        self.concat_X = np.concatenate([self.X, self.sp.reshape(-1, 1), self.exp], axis=1)
        self.feat_names = []
        self.feat_names.extend([f"HA_{i}" for i in range(HA_HB_DIM)])
        self.feat_names.extend([f"HB_{i}" for i in range(HA_HB_DIM)])
        self.feat_names.extend([f"Inter_{i}" for i in range(DIM_INTER)])
        self.feat_names.append("Species_Code")
        self.feat_names.append("Exposure_Day")
        assert len(self.feat_names) == TOTAL_INPUT_DIM

    def compute_global_importance(self, target_cls):
        n = self.sample_num
        shap_vals = np.zeros((n, TOTAL_INPUT_DIM))
        for idx in range(n):
            x_feat = torch.tensor(self.X[idx:idx+1], dtype=torch.float32, device=self.device, requires_grad=True)
            x_sp = torch.tensor([self.sp[idx]], dtype=torch.long, device=self.device)
            x_exp = torch.tensor(self.exp[idx:idx+1], dtype=torch.float32, device=self.device, requires_grad=True)
            logits, _ = self.model(x_feat, x_sp, x_exp)
            target_score = logits[0, target_cls]
            self.model.zero_grad()
            target_score.backward()
            grad_feat = x_feat.grad.cpu().numpy()[0]
            grad_exp = x_exp.grad.cpu().numpy()[0]
            grad_sp = np.array([0.0])
            full_grad = np.concatenate([grad_feat, grad_sp, grad_exp])
            shap_vals[idx] = full_grad * self.concat_X[idx]
        global_abs_shap = np.mean(np.abs(shap_vals), axis=0)
        return global_abs_shap

def calc_all_modal_contribution(global_abs_shap, modal_to_hidden):
    ha_shap  = global_abs_shap[:HA_HB_DIM]
    hb_shap  = global_abs_shap[HA_HB_DIM : HA_HB_DIM*2]
    inter_shap = global_abs_shap[HA_HB_DIM*2 : HA_HB_DIM*2 + DIM_INTER]
    species_shap = global_abs_shap[-2]
    exposure_shap = global_abs_shap[-1]

    modal_A = np.zeros(6)
    modal_B = np.zeros(6)
    for m in range(6):
        modal_A[m] = np.sum(modal_to_hidden[m] * ha_shap)
        modal_B[m] = np.sum(modal_to_hidden[m] * hb_shap)
    inner_total = modal_A + modal_B

    inter_total = np.sum(np.abs(inter_shap))
    species_total = np.abs(species_shap)
    exposure_total = np.abs(exposure_shap)
    extra_total = np.array([inter_total, species_total, exposure_total])

    all_scores = np.concatenate([inner_total, extra_total])
    sum_all = np.sum(all_scores)
    all_ratio = all_scores / sum_all

    df = pd.DataFrame({
        "Feature_Modal": ALL_MODAL_NAMES,
        "Total_Contribution_Score": all_scores,
        "Contribution_Percent(%)": np.round(all_ratio * 100, 2)
    })
    return df

def plot_all_modal_figure(df_modal, target_cls, cls_name, save_folder):
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['font.size'] = 11

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18,7))
    names = df_modal["Feature_Modal"].tolist()
    score = df_modal["Total_Contribution_Score"].tolist()
    pct = df_modal["Contribution_Percent(%)"].tolist()

    ax1.barh(names, score, color="#4472C4")
    ax1.invert_yaxis()
    ax1.set_xlabel("Total Contribution Score", fontsize=12)
    ax1.set_title(f"Total Contribution of Nine Feature Modalities | {cls_name.replace('_',' ')}", fontsize=13, pad=12)
    ax1.grid(axis='x', alpha=0.3, linestyle='--')

    ax2.pie(pct, labels=names, autopct="%.2f%%", textprops={"fontsize":10})
    ax2.set_title(f"Proportion of Feature Contribution | {cls_name.replace('_',' ')}", fontsize=13, pad=12)

    plt.tight_layout()
    save_name = f"modal_contribution_{target_cls}_{cls_name}.png"
    full_save_path = os.path.join(save_folder, save_name)
    plt.savefig(full_save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Figure saved to: {full_save_path}")

current_cls_name = cls_desc[TARGET_CLS]

modal_weight_matrix = load_fusion_weight_breakdown(device=device)
print("Fusion network projection weight matrix loaded (6 modal × 192 hidden dimension)")

explainer = GradSHAPExplainer(
    model=model,
    X_test=X_test,
    sp_test=sp_test,
    exp_test=exp_test,
    device=device,
    sample_num=SHAP_SAMPLE_NUM
)

global_shap = explainer.compute_global_importance(target_cls=TARGET_CLS)

modal_result_df = calc_all_modal_contribution(global_shap, modal_weight_matrix)

print(f"\n==================== {current_cls_name.replace('_',' ')} Global Contribution of Nine Feature Modalities ====================")
print(modal_result_df)
csv_save_name = f"modal_contribution_{TARGET_CLS}_{current_cls_name}.csv"
csv_full_path = os.path.join(save_dir, csv_save_name)
modal_result_df.to_csv(csv_full_path, index=False)
print(f"Contribution table exported to: {csv_full_path}")

plot_all_modal_figure(modal_result_df, TARGET_CLS, current_cls_name, save_dir)

print("\n==================== All analysis finished ====================")
print(f"All outputs are stored in folder: {save_dir}")
print(f"Current target class file prefix: modal_contribution_{TARGET_CLS}_{current_cls_name}")

In [ ]:
#Global modal self-attention weight

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
import os

plt.rcParams["font.family"] = ["Arial"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 10
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.02

MODAL_NAMES = [
    "F1 Basic Physicochemical",
    "F2 PCA Morgan Fingerprint",
    "F3 Scaled Physicochemical",
    "F4 3D Geometric Descriptors",
    "HB Hydrogen Bond Count",
    "Assay Experimental Probability"
]

LABEL_DICT = {
    0: "Antagonistic_Effect",
    1: "Synergistic_Effect",
    2: "Not_Specified_Interaction",
    3: "Additive_Effect",
    4: "Independent_Effect"
}
LABEL_TITLE = {
    0: "Antagonistic Effect",
    1: "Synergistic Effect",
    2: "Not Specified Interaction",
    3: "Additive Effect",
    4: "Independent Effect"
}
NUM_CLASSES = 5

OUTPUT_FOLDER = "reproduction_Attention_Heatmaps"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

def get_modal_self_attn(fusion, modal_input):
    bs = modal_input[0].size(0)
    hidden_list = []
    for idx, m in enumerate(modal_input):
        h = fusion.projs[idx](m)
        h = fusion.ln_proj(h)
        hidden_list.append(h)
    stack = torch.stack(hidden_list, dim=1)

    qkv = fusion.qkv_global(stack).reshape(
        bs, fusion.num_modal, 3, fusion.head, fusion.head_dim
    ).permute(2,0,3,1,4)
    q_g, k_g, v_g = qkv[0], qkv[1], qkv[2]
    score = torch.matmul(q_g, k_g.transpose(-1,-2)) / np.sqrt(fusion.head_dim)
    global_attn = F.softmax(score, dim=-1)

    global_attn_detach = global_attn.detach().cpu()
    attn_mat = global_attn_detach[:, 0, :, :].mean(dim=0).numpy()
    modal_mean_weight = attn_mat.mean(axis=0)
    return attn_mat, modal_mean_weight

def plot_class_global_avg_heatmap(target_cls_id, sample_num=100):
    fusion_model.eval()
    cls_name_file = LABEL_DICT[target_cls_id]
    cls_name_title = LABEL_TITLE[target_cls_id]
    target_df = df[df["label"] == target_cls_id].reset_index(drop=True)
    total_compound = len(target_df)
    extract_n = min(sample_num, total_compound)
    use_df = target_df.head(extract_n)
    print(f"\n===== Processing {cls_name_title} | Sampling {extract_n}/{total_compound} compounds =====")

    attn_sum = np.zeros((6,6))
    modal_weight_sum = np.zeros(6)
    with torch.no_grad():
        for _, row in use_df.iterrows():
            f1a, f2a_raw, f3a, f4a, hb_a = extract_mol_features_raw(row["SMILES1"])
            f2a = pca.transform(f2a_raw.reshape(1,-1))[0]
            assayA = row[["SMILES1-assay1","SMILES1-assay2","SMILES1-assay3"]].values.astype(np.float32)
            modA = [
                torch.from_numpy(f1a).float().unsqueeze(0).to(device),
                torch.from_numpy(f2a).float().unsqueeze(0).to(device),
                torch.from_numpy(f3a).float().unsqueeze(0).to(device),
                torch.from_numpy(f4a).float().unsqueeze(0).to(device),
                torch.from_numpy(hb_a).float().unsqueeze(0).to(device),
                torch.from_numpy(assayA).float().unsqueeze(0).to(device)
            ]
            attn_mat, modal_w = get_modal_self_attn(fusion_model, modA)
            attn_sum += attn_mat
            modal_weight_sum += modal_w
    avg_attn_mat = attn_sum / extract_n
    avg_modal_w = modal_weight_sum / extract_n

    fig, ax = plt.subplots(figsize=(9,7), dpi=300)
    sns.heatmap(
        avg_attn_mat,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=MODAL_NAMES,
        yticklabels=MODAL_NAMES,
        ax=ax
    )
    ax.set_title(f"Global Average Self-Attention Heatmap of Six Feature Modalities | {cls_name_title} (n={extract_n})", fontsize=12)
    ax.set_xlabel("Target Attended Feature Modality", fontsize=11)
    ax.set_ylabel("Query Feature Modality", fontsize=11)
    plt.xticks(rotation=30, ha="right")

    file_base = os.path.join(OUTPUT_FOLDER, f"Heatmap_{cls_name_file}_n{extract_n}")
    plt.savefig(f"{file_base}.tiff", dpi=300)
    plt.savefig(f"{file_base}.pdf", format="pdf")
    plt.close(fig)
    print(f"Export finished: {file_base} (.tiff / .pdf)")

    print(f"\n[Global Average Attention Weights - {cls_name_title}]")
    for feat_name, w_val in zip(MODAL_NAMES, avg_modal_w):
        print(f"{feat_name}: {w_val:.4f}")
    top_feature_idx = np.argmax(avg_modal_w)
    print(f"Dominant discriminative feature of this class: {MODAL_NAMES[top_feature_idx]}, Mean Attention = {avg_modal_w[top_feature_idx]:.4f}\n")
    return avg_attn_mat, avg_modal_w

if __name__ == "__main__":
    fusion_model.eval()
    MAX_SAMPLE_PER_CLASS = 100
    for cls_id in range(NUM_CLASSES):
        group_data = df[df["label"] == cls_id]
        if len(group_data) == 0:
            print(f"\nWarning: No available samples for {LABEL_TITLE[cls_id]}, skip export")
            continue
        avg_matrix, avg_weight = plot_class_global_avg_heatmap(target_cls_id=cls_id, sample_num=MAX_SAMPLE_PER_CLASS)
